# ARC Prize 2026 — Entrenamiento LoRA (kernel de Kaggle, GPU L4×4)

Corre en GPU de Kaggle **L4×4 (96GB, internet OFF obligatorio)**. Pipeline: construir dataset
augmentado → fine-tune SFT con LoRA → guardar adaptador en `/kaggle/working` (luego se publica
como dataset para el kernel de inferencia `submit.ipynb`).

**Adjuntar a este kernel (todo offline):**
- La competencia `arc-prize-2026-arc-agi-2` (datos).
- El código del repo como dataset (con `src/` y `scripts/`).
- El **modelo base** desde Kaggle Models (p.ej. Qwen2.5-7B-Instruct) — NO se descarga de HF.

> L4×4 exige internet deshabilitado y consume quota al doble; solo disponible adjuntado a esta competencia.

In [ ]:
# OFFLINE (L4x4 exige internet OFF): nada de pip install ni descargas HF.
# La imagen GPU de Kaggle ya trae torch/transformers/peft/accelerate.
import os, sys, glob, json

OUT_DIR = '/kaggle/working/lora-adapter'
DATA_JSONL = '/kaggle/working/finetune_train.jsonl'

SMOKE = True        # <<< run de validacion rapido. Pon False para el entrenamiento real.
N_AUG = 2 if SMOKE else 8           # augmentaciones (D4+color) por tarea REAL
N_SYNTH = 200 if SMOKE else 20000   # nº de tareas SINTÉTICAS
EPOCHS = 1.0
BATCH = 4
GRAD_ACCUM = 8
MAX_STEPS = 60 if SMOKE else -1     # -1 = sin limite (usa EPOCHS)

# Localiza código (src/ y scripts/)
CODE_DIR = None
for c in glob.glob('/kaggle/input/*'):
    if os.path.isdir(os.path.join(c, 'src', 'arc')):
        CODE_DIR = c; break
assert CODE_DIR, 'Adjunta el repo (con src/ y scripts/) como dataset.'
sys.path.insert(0, os.path.join(CODE_DIR, 'src'))
sys.path.insert(0, os.path.join(CODE_DIR, 'scripts'))

# Localiza el MODELO BASE adjunto (Kaggle Models/Dataset con config.json). Sin descargas.
MODEL_DIR = None
for cfg in glob.glob('/kaggle/input/**/config.json', recursive=True):
    if 'adapter_config.json' not in os.listdir(os.path.dirname(cfg)):
        MODEL_DIR = os.path.dirname(cfg); break
assert MODEL_DIR, 'Adjunta el modelo base (Kaggle Models) — p.ej. Qwen2.5-7B-Instruct.'
COMP_DIR = '/kaggle/input/arc-prize-2026-arc-agi-2'
print('SMOKE =', SMOKE, '| CODE_DIR =', CODE_DIR); print('MODEL_DIR =', MODEL_DIR)

In [ ]:
# 1) Construir dataset = tareas REALES augmentadas + tareas SINTÉTICAS
from arc.data import load_split
from arc.synth import generate_dataset
from build_finetune_dataset import samples_for_task

n = 0
with open(DATA_JSONL, 'w', encoding='utf-8') as f:
    # reales (D4 + color + leave-one-out)
    for task in load_split('training', data_dir=COMP_DIR).values():
        for s in samples_for_task(task, n_aug=N_AUG, max_cells=900, sep=''):
            f.write(json.dumps(s) + '\n'); n += 1
    # sintéticas (la regla es consistente por tarea; el modelo aprende a inducirla)
    for task in generate_dataset(N_SYNTH, seed=0):
        for s in samples_for_task(task, n_aug=1, max_cells=900, sep=''):
            f.write(json.dumps(s) + '\n'); n += 1
print('Muestras totales:', n, '->', DATA_JSONL)

In [ ]:
# 2) Fine-tune SFT con LoRA (reutiliza train_lora.py). Modelo base desde MODEL_DIR (offline).
import torch
from transformers import Trainer, TrainingArguments
from arc.model import load_model, build_lora
from train_lora import ArcSFTDataset, PadCollator, read_jsonl

rows = list(read_jsonl(DATA_JSONL))
model, tok = load_model(MODEL_DIR, device_map='cuda')
model = build_lora(model, r=16)
model.print_trainable_parameters()

ds = ArcSFTDataset(rows, tok, max_seq_len=2048)
targs = TrainingArguments(
    output_dir='/kaggle/working/ckpt', num_train_epochs=EPOCHS, max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=1e-4, lr_scheduler_type='cosine', warmup_ratio=0.03,
    logging_steps=10, save_strategy='no', bf16=True, report_to='none')
trainer = Trainer(model=model, args=targs, train_dataset=ds,
                  data_collator=PadCollator(pad_id=tok.pad_token_id))
trainer.train()

In [ ]:
# 3) Guardar adaptador LoRA + tokenizer (sera la salida del kernel)
os.makedirs(OUT_DIR, exist_ok=True)
model.save_pretrained(OUT_DIR)
tok.save_pretrained(OUT_DIR)
print('LoRA guardado en', OUT_DIR)
print(os.listdir(OUT_DIR))

## Tras ejecutar
1. Guarda la salida del kernel (`/kaggle/working/lora-adapter`) como **Dataset** (p.ej. `arc-agi2-model`).
2. Adjunta ese dataset (y el modelo base) al kernel de inferencia `submit.ipynb` (internet OFF).
3. Valida en `evaluation` con `scripts/evaluate.py` antes de enviar.